In [1]:
import requests
import pandas as pd

In [2]:
APPLE_CIK = "0000320193"

url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{APPLE_CIK}.json"

url

'https://data.sec.gov/api/xbrl/companyfacts/CIK0000320193.json'

In [3]:
import getpass

SEC_USER_AGENT = getpass.getpass(
    "Enter SEC User-Agent (project name + contact email): "
)

headers = {
    "User-Agent": SEC_USER_AGENT
}

response = requests.get(
    url,
    headers=headers
)

response.raise_for_status()

In [4]:
apple_data = response.json()

In [5]:
type(apple_data)

dict

In [6]:
apple_data.keys()

dict_keys(['cik', 'entityName', 'facts'])

In [7]:
apple_data["entityName"]

'Apple Inc.'

In [8]:
apple_data["cik"]

320193

In [9]:
type(apple_data["facts"])

dict

In [10]:
apple_data["facts"].keys()

dict_keys(['dei', 'us-gaap'])

In [11]:
us_gaap = apple_data["facts"]["us-gaap"]

In [12]:
type(us_gaap)

dict

In [13]:
len(us_gaap)

503

In [14]:
revenue_concepts = [
    key
    for key in us_gaap.keys()
    if "Revenue" in key
]

In [15]:
revenue_concepts

['ContractWithCustomerLiabilityRevenueRecognized',
 'DeferredRevenueCurrent',
 'DeferredRevenueNoncurrent',
 'IncreaseDecreaseInDeferredRevenue',
 'RevenueFromContractWithCustomerExcludingAssessedTax',
 'Revenues',
 'SalesRevenueNet',
 'SalesRevenueServicesGross']

In [16]:
revenue_concept = us_gaap["RevenueFromContractWithCustomerExcludingAssessedTax"]

In [17]:
type(revenue_concept)

dict

In [18]:
print("Label:")
print(revenue_concept["label"])

print("\nDescription:")
print(revenue_concept["description"])

print("\nAvailable units:")
print(list(revenue_concept["units"].keys()))

Label:
Revenue from Contract with Customer, Excluding Assessed Tax

Description:
Amount, excluding tax collected from customer, of revenue from satisfaction of performance obligation by transferring promised good or service to customer. Tax collected from customer is tax assessed by governmental authority that is both imposed on and concurrent with specific revenue-producing transaction, including, but not limited to, sales, use, value added and excise.

Available units:
['USD']


In [19]:
revenue_usd = revenue_concept["units"]["USD"]

In [20]:
type(revenue_usd)

list

In [21]:
len(revenue_usd)

117

In [22]:
revenue_usd[0]

{'start': '2016-09-25',
 'end': '2017-09-30',
 'val': 229234000000,
 'accn': '0000320193-19-000119',
 'fy': 2019,
 'fp': 'FY',
 'form': '10-K',
 'filed': '2019-10-31',
 'frame': 'CY2017'}

In [23]:
revenue_df = pd.DataFrame(revenue_usd)

In [24]:
revenue_df.head()

,start,end,val,accn,fy,fp,form,filed,frame
0,2016-09-25,2017-09-30,229234000000,0000320193-19-000119,2019,FY,10-K,2019-10-31,CY2017
1,2017-10-01,2017-12-30,88293000000,0000320193-19-000010,2019,Q1,10-Q,2019-01-30,NaN
2,2017-10-01,2017-12-30,88293000000,0000320193-19-000119,2019,FY,10-K,2019-10-31,CY2017Q4
3,2017-10-01,2018-03-31,149430000000,0000320193-19-000066,2019,Q2,10-Q,2019-05-01,NaN
4,2017-12-31,2018-03-31,61137000000,0000320193-19-000066,2019,Q2,10-Q,2019-05-01,NaN


In [25]:
revenue_df.columns

Index(['start', 'end', 'val', 'accn', 'fy', 'fp', 'form', 'filed', 'frame'], dtype='str')

In [26]:
annual_revenue = revenue_df[
    (revenue_df["form"] == "10-K") &
    (revenue_df["fp"] == "FY")
].copy()

In [27]:
annual_revenue["start"] = pd.to_datetime(annual_revenue["start"])
annual_revenue["end"] = pd.to_datetime(annual_revenue["end"])

In [28]:
annual_revenue["period_days"] = (
    annual_revenue["end"] - annual_revenue["start"]
).dt.days

In [29]:
annual_revenue[
    ["start", "end", "val", "fy", "filed", "period_days"]
].tail(20)

,start,end,val,fy,filed,period_days
32,2019-06-30,2019-09-28,64040000000,2020,2020-10-30,90
34,2019-09-29,2019-12-28,91819000000,2020,2020-10-30,90
39,2019-12-29,2020-03-28,58313000000,2020,2020-10-30,90
44,2020-03-29,2020-06-27,59685000000,2020,2020-10-30,90
46,2019-09-29,2020-09-26,274515000000,2020,2020-10-30,363
47,2019-09-29,2020-09-26,274515000000,2021,2021-10-29,363
48,2019-09-29,2020-09-26,274515000000,2022,2022-10-28,363
49,2020-06-28,2020-09-26,64698000000,2020,2020-10-30,90
60,2020-09-27,2021-09-25,365817000000,2021,2021-10-29,363
61,2020-09-27,2021-09-25,365817000000,2022,2022-10-28,363


In [30]:
full_year_revenue = annual_revenue[
    annual_revenue["period_days"] > 300
].copy()

In [31]:
full_year_revenue[
    ["start", "end", "val", "fy", "filed", "period_days"]
]

,start,end,val,fy,filed,period_days
0,2016-09-25,2017-09-30,229234000000,2019,2019-10-31,370
9,2017-10-01,2018-09-29,265595000000,2019,2019-10-31,363
10,2017-10-01,2018-09-29,265595000000,2020,2020-10-30,363
28,2018-09-30,2019-09-28,260174000000,2019,2019-10-31,363
29,2018-09-30,2019-09-28,260174000000,2020,2020-10-30,363
30,2018-09-30,2019-09-28,260174000000,2021,2021-10-29,363
46,2019-09-29,2020-09-26,274515000000,2020,2020-10-30,363
47,2019-09-29,2020-09-26,274515000000,2021,2021-10-29,363
48,2019-09-29,2020-09-26,274515000000,2022,2022-10-28,363
60,2020-09-27,2021-09-25,365817000000,2021,2021-10-29,363


In [32]:
full_year_revenue = full_year_revenue.sort_values("filed")

In [33]:
clean_annual_revenue = full_year_revenue.drop_duplicates(
    subset=["start", "end"],
    keep="last"
).copy()

In [34]:
clean_annual_revenue[
    ["start", "end", "val", "fy", "filed", "period_days"]
]

,start,end,val,fy,filed,period_days
0,2016-09-25,2017-09-30,229234000000,2019,2019-10-31,370
10,2017-10-01,2018-09-29,265595000000,2020,2020-10-30,363
30,2018-09-30,2019-09-28,260174000000,2021,2021-10-29,363
48,2019-09-29,2020-09-26,274515000000,2022,2022-10-28,363
62,2020-09-27,2021-09-25,365817000000,2023,2023-11-03,363
75,2021-09-26,2022-09-24,394328000000,2024,2024-11-01,363
88,2022-09-25,2023-09-30,383285000000,2025,2025-10-31,370
100,2023-10-01,2024-09-28,391035000000,2025,2025-10-31,363
111,2024-09-29,2025-09-27,416161000000,2025,2025-10-31,363


In [35]:
clean_annual_revenue["fiscal_year"] = clean_annual_revenue["end"].dt.year

In [36]:
clean_annual_revenue[
    ["fiscal_year", "start", "end", "val", "filed"]
]

,fiscal_year,start,end,val,filed
0,2017,2016-09-25,2017-09-30,229234000000,2019-10-31
10,2018,2017-10-01,2018-09-29,265595000000,2020-10-30
30,2019,2018-09-30,2019-09-28,260174000000,2021-10-29
48,2020,2019-09-29,2020-09-26,274515000000,2022-10-28
62,2021,2020-09-27,2021-09-25,365817000000,2023-11-03
75,2022,2021-09-26,2022-09-24,394328000000,2024-11-01
88,2023,2022-09-25,2023-09-30,383285000000,2025-10-31
100,2024,2023-10-01,2024-09-28,391035000000,2025-10-31
111,2025,2024-09-29,2025-09-27,416161000000,2025-10-31


In [37]:
apple_revenue_2021_2025 = clean_annual_revenue[
    clean_annual_revenue["fiscal_year"].between(2021, 2025)
].copy()

In [38]:
apple_revenue_2021_2025["revenue_billions"] = (
    apple_revenue_2021_2025["val"] / 1_000_000_000
)

In [39]:
apple_revenue_2021_2025 = apple_revenue_2021_2025.reset_index(drop=True)

In [40]:
apple_revenue_2021_2025[
    ["fiscal_year", "revenue_billions"]
]

,fiscal_year,revenue_billions
0,2021,365.817
1,2022,394.328
2,2023,383.285
3,2024,391.035
4,2025,416.161


In [41]:
apple_revenue_clean = apple_revenue_2021_2025[
    ["fiscal_year", "start", "end", "val", "filed"]
].copy()

In [42]:
apple_revenue_clean = apple_revenue_clean.rename(
    columns={
        "start": "period_start",
        "end": "period_end",
        "val": "value_usd"
    }
)

In [43]:
apple_revenue_clean["company"] = "Apple"
apple_revenue_clean["ticker"] = "AAPL"
apple_revenue_clean["metric"] = "Revenue"
apple_revenue_clean["value_billions"] = (
    apple_revenue_clean["value_usd"] / 1_000_000_000
)

In [44]:
apple_revenue_clean

,fiscal_year,period_start,period_end,value_usd,filed,company,ticker,metric,value_billions
0,2021,2020-09-27,2021-09-25,365817000000,2023-11-03,Apple,AAPL,Revenue,365.817
1,2022,2021-09-26,2022-09-24,394328000000,2024-11-01,Apple,AAPL,Revenue,394.328
2,2023,2022-09-25,2023-09-30,383285000000,2025-10-31,Apple,AAPL,Revenue,383.285
3,2024,2023-10-01,2024-09-28,391035000000,2025-10-31,Apple,AAPL,Revenue,391.035
4,2025,2024-09-29,2025-09-27,416161000000,2025-10-31,Apple,AAPL,Revenue,416.161


In [45]:
clean_annual_revenue = (
    full_year_revenue
    .sort_values("filed")
    .drop_duplicates(
        subset=["start", "end"],
        keep="first"
    )
    .copy()
)

In [46]:
clean_annual_revenue["fiscal_year"] = clean_annual_revenue["end"].dt.year

In [47]:
apple_revenue_2021_2025 = clean_annual_revenue[
    clean_annual_revenue["fiscal_year"].between(2021, 2025)
].copy()

In [48]:
apple_revenue_2021_2025 = clean_annual_revenue[
    clean_annual_revenue["fiscal_year"].between(2021, 2025)
].copy()

apple_revenue_2021_2025 = apple_revenue_2021_2025.reset_index(drop=True)

In [49]:
apple_revenue_clean = apple_revenue_2021_2025[
    ["fiscal_year", "start", "end", "val", "filed"]
].copy()

apple_revenue_clean = apple_revenue_clean.rename(
    columns={
        "start": "period_start",
        "end": "period_end",
        "val": "value_usd"
    }
)

apple_revenue_clean["company"] = "Apple"
apple_revenue_clean["ticker"] = "AAPL"
apple_revenue_clean["metric"] = "Revenue"

apple_revenue_clean["value_billions"] = (
    apple_revenue_clean["value_usd"] / 1_000_000_000
)

In [50]:
apple_revenue_clean

,fiscal_year,period_start,period_end,value_usd,filed,company,ticker,metric,value_billions
0,2021,2020-09-27,2021-09-25,365817000000,2021-10-29,Apple,AAPL,Revenue,365.817
1,2022,2021-09-26,2022-09-24,394328000000,2022-10-28,Apple,AAPL,Revenue,394.328
2,2023,2022-09-25,2023-09-30,383285000000,2023-11-03,Apple,AAPL,Revenue,383.285
3,2024,2023-10-01,2024-09-28,391035000000,2024-11-01,Apple,AAPL,Revenue,391.035
4,2025,2024-09-29,2025-09-27,416161000000,2025-10-31,Apple,AAPL,Revenue,416.161


In [51]:
def extract_annual_metric(
    company_data,
    concept_name,
    company,
    ticker,
    metric
):
    if isinstance(concept_name, str):
        concept_names = [concept_name]
    else:
        concept_names = concept_name

    observations = []

    for name in concept_names:
        concept_observations = (
            company_data["facts"]["us-gaap"][name]["units"]["USD"]
        )

        observations.extend(concept_observations)

    df = pd.DataFrame(observations)

    df["start"] = pd.to_datetime(df["start"])
    df["end"] = pd.to_datetime(df["end"])

    df = df[
        (df["form"] == "10-K") &
        (df["fp"] == "FY")
    ].copy()

    df["period_days"] = (
        df["end"] - df["start"]
    ).dt.days

    df = df[
        df["period_days"] > 300
    ].copy()

    df["fiscal_year"] = df["end"].dt.year

    df = df[
    df["fiscal_year"].between(2021, 2025)
    ].copy()

    restatement_check = (
        df
        .groupby(["start", "end"])["val"]
        .nunique()
        .reset_index(name="unique_values")
    )

    restated_periods = restatement_check[
        restatement_check["unique_values"] > 1
    ]

    if not restated_periods.empty:
        raise ValueError(
            f"Conflicting values found for {company} - {metric}"
        )

    df = (
        df
        .sort_values("filed")
        .drop_duplicates(
            subset=["start", "end"],
            keep="first"
        )
        .copy()
    )

    df = df[
    ["fiscal_year", "start", "end", "val", "filed"]
    ].copy()

    df = df.rename(
    columns={
        "start": "period_start",
        "end": "period_end",
        "val": "value_usd"
        }
    )

    df["company"] = company
    df["ticker"] = ticker
    df["metric"] = metric

    df["value_billions"] = (
    df["value_usd"] / 1_000_000_000
    )

    df = df.reset_index(drop=True)

    return df

In [52]:
operating_income_observations = (
    apple_data["facts"]["us-gaap"]["OperatingIncomeLoss"]["units"]["USD"]
)

operating_income_df = pd.DataFrame(operating_income_observations)

operating_income_df["start"] = pd.to_datetime(
    operating_income_df["start"]
)
operating_income_df["end"] = pd.to_datetime(
    operating_income_df["end"]
)

operating_income_df = operating_income_df[
    (operating_income_df["form"] == "10-K") &
    (operating_income_df["fp"] == "FY")
].copy()

operating_income_df["period_days"] = (
    operating_income_df["end"] -
    operating_income_df["start"]
).dt.days

operating_income_df = operating_income_df[
    operating_income_df["period_days"] > 300
].copy()

In [53]:
operating_income_conflicts = (
    operating_income_df
    .groupby(["start", "end"])["val"]
    .nunique()
    .reset_index(name="unique_values")
)

operating_income_conflicts[
    operating_income_conflicts["unique_values"] > 1
]

,start,end,unique_values
1,2007-09-30,2008-09-27,2
2,2008-09-28,2009-09-26,2


In [54]:
test_revenue = extract_annual_metric(
    company_data=apple_data,
    concept_name="RevenueFromContractWithCustomerExcludingAssessedTax",
    company="Apple",
    ticker="AAPL",
    metric="Revenue"
)

In [55]:
test_revenue

,fiscal_year,period_start,period_end,value_usd,filed,company,ticker,metric,value_billions
0,2021,2020-09-27,2021-09-25,365817000000,2021-10-29,Apple,AAPL,Revenue,365.817
1,2022,2021-09-26,2022-09-24,394328000000,2022-10-28,Apple,AAPL,Revenue,394.328
2,2023,2022-09-25,2023-09-30,383285000000,2023-11-03,Apple,AAPL,Revenue,383.285
3,2024,2023-10-01,2024-09-28,391035000000,2024-11-01,Apple,AAPL,Revenue,391.035
4,2025,2024-09-29,2025-09-27,416161000000,2025-10-31,Apple,AAPL,Revenue,416.161


In [56]:
operating_income_concepts = [
    key
    for key in us_gaap.keys()
    if "OperatingIncome" in key
]

operating_income_concepts

['OperatingIncomeLoss', 'SegmentReportingInformationOperatingIncomeLoss']

In [57]:
operating_income_concept = us_gaap["OperatingIncomeLoss"]

In [58]:
print("Label:")
print(operating_income_concept["label"])

print("\nDescription:")
print(operating_income_concept["description"])

print("\nAvailable units:")
print(list(operating_income_concept["units"].keys()))

Label:
Operating Income (Loss)

Description:
The net result for the period of deducting operating expenses from operating revenues.

Available units:
['USD']


In [59]:
apple_operating_income = extract_annual_metric(
    company_data=apple_data,
    concept_name="OperatingIncomeLoss",
    company="Apple",
    ticker="AAPL",
    metric="Operating Income"
)

In [60]:
apple_operating_income

,fiscal_year,period_start,period_end,value_usd,filed,company,ticker,metric,value_billions
0,2021,2020-09-27,2021-09-25,108949000000,2021-10-29,Apple,AAPL,Operating Income,108.949
1,2022,2021-09-26,2022-09-24,119437000000,2022-10-28,Apple,AAPL,Operating Income,119.437
2,2023,2022-09-25,2023-09-30,114301000000,2023-11-03,Apple,AAPL,Operating Income,114.301
3,2024,2023-10-01,2024-09-28,123216000000,2024-11-01,Apple,AAPL,Operating Income,123.216
4,2025,2024-09-29,2025-09-27,133050000000,2025-10-31,Apple,AAPL,Operating Income,133.050


In [61]:
net_income_concepts = [
    key
    for key in us_gaap.keys()
    if "NetIncome" in key
]

net_income_concepts

['NetIncomeLoss',
 'OtherComprehensiveIncomeLossReclassificationAdjustmentForSaleOfSecuritiesIncludedInNetIncomeNetOfTax',
 'OtherComprehensiveIncomeLossReclassificationAdjustmentOnDerivativesIncludedInNetIncomeNetOfTax',
 'OtherComprehensiveIncomeLossReclassificationAdjustmentOnDerivativesIncludedInNetIncomeTax',
 'OtherComprehensiveIncomeReclassificationAdjustmentOnDerivativesIncludedInNetIncomeNetOfTax',
 'OtherComprehensiveIncomeReclassificationAdjustmentOnDerivativesIncludedInNetIncomeTax']

In [62]:
net_income_concept = us_gaap["NetIncomeLoss"]

In [63]:
print("Label:")
print(net_income_concept["label"])

print("\nDescription:")
print(net_income_concept["description"])

print("\nAvailable units:")
print(list(net_income_concept["units"].keys()))

Label:
Net Income (Loss) Attributable to Parent

Description:
The portion of profit or loss for the period, net of income taxes, which is attributable to the parent.

Available units:
['USD']


In [64]:
apple_net_income = extract_annual_metric(
    company_data=apple_data,
    concept_name="NetIncomeLoss",
    company="Apple",
    ticker="AAPL",
    metric="Net Income"
)

In [65]:
apple_net_income

,fiscal_year,period_start,period_end,value_usd,filed,company,ticker,metric,value_billions
0,2021,2020-09-27,2021-09-25,94680000000,2021-10-29,Apple,AAPL,Net Income,94.680
1,2022,2021-09-26,2022-09-24,99803000000,2022-10-28,Apple,AAPL,Net Income,99.803
2,2023,2022-09-25,2023-09-30,96995000000,2023-11-03,Apple,AAPL,Net Income,96.995
3,2024,2023-10-01,2024-09-28,93736000000,2024-11-01,Apple,AAPL,Net Income,93.736
4,2025,2024-09-29,2025-09-27,112010000000,2025-10-31,Apple,AAPL,Net Income,112.010


In [66]:
rd_concepts = [
    key
    for key in us_gaap.keys()
    if "ResearchAndDevelopment" in key
]

rd_concepts

['ResearchAndDevelopmentExpense']

In [67]:
rd_concept = us_gaap["ResearchAndDevelopmentExpense"]

In [68]:
print("Label:")
print(rd_concept["label"])

print("\nDescription:")
print(rd_concept["description"])

print("\nAvailable units:")
print(list(rd_concept["units"].keys()))

Label:
Research and Development Expense

Description:
The aggregate costs incurred (1) in a planned search or critical investigation aimed at discovery of new knowledge with the hope that such knowledge will be useful in developing a new product or service, a new process or technique, or in bringing about a significant improvement to an existing product or process; or (2) to translate research findings or other knowledge into a plan or design for a new product or process or for a significant improvement to an existing product or process whether intended for sale or the entity's use, during the reporting period charged to research and development projects, including the costs of developing computer software up to the point in time of achieving technological feasibility, and costs allocated in accounting for a business combination to in-process projects deemed to have no alternative future use.

Available units:
['USD']


In [69]:
apple_rd = extract_annual_metric(
    company_data=apple_data,
    concept_name="ResearchAndDevelopmentExpense",
    company="Apple",
    ticker="AAPL",
    metric="R&D Expense"
)

In [70]:
apple_rd

,fiscal_year,period_start,period_end,value_usd,filed,company,ticker,metric,value_billions
0,2021,2020-09-27,2021-09-25,21914000000,2021-10-29,Apple,AAPL,R&D Expense,21.914
1,2022,2021-09-26,2022-09-24,26251000000,2022-10-28,Apple,AAPL,R&D Expense,26.251
2,2023,2022-09-25,2023-09-30,29915000000,2023-11-03,Apple,AAPL,R&D Expense,29.915
3,2024,2023-10-01,2024-09-28,31370000000,2024-11-01,Apple,AAPL,R&D Expense,31.370
4,2025,2024-09-29,2025-09-27,34550000000,2025-10-31,Apple,AAPL,R&D Expense,34.550


In [71]:
cash_flow_concepts = [
    key
    for key in us_gaap.keys()
    if "OperatingActivities" in key
]

cash_flow_concepts

['NetCashProvidedByUsedInOperatingActivities',
 'NetCashProvidedByUsedInOperatingActivitiesContinuingOperations']

In [72]:
operating_cash_flow_concept = us_gaap[
    "NetCashProvidedByUsedInOperatingActivities"
]

In [73]:
print("Label:")
print(operating_cash_flow_concept["label"])

print("\nDescription:")
print(operating_cash_flow_concept["description"])

print("\nAvailable units:")
print(list(operating_cash_flow_concept["units"].keys()))

Label:
Net Cash Provided by (Used in) Operating Activities

Description:
Amount of cash inflow (outflow) from operating activities, including discontinued operations. Operating activity cash flows include transactions, adjustments, and changes in value not defined as investing or financing activities.

Available units:
['USD']


In [74]:
apple_operating_cash_flow = extract_annual_metric(
    company_data=apple_data,
    concept_name="NetCashProvidedByUsedInOperatingActivities",
    company="Apple",
    ticker="AAPL",
    metric="Operating Cash Flow"
)

In [75]:
apple_operating_cash_flow

,fiscal_year,period_start,period_end,value_usd,filed,company,ticker,metric,value_billions
0,2021,2020-09-27,2021-09-25,104038000000,2021-10-29,Apple,AAPL,Operating Cash Flow,104.038
1,2022,2021-09-26,2022-09-24,122151000000,2022-10-28,Apple,AAPL,Operating Cash Flow,122.151
2,2023,2022-09-25,2023-09-30,110543000000,2023-11-03,Apple,AAPL,Operating Cash Flow,110.543
3,2024,2023-10-01,2024-09-28,118254000000,2024-11-01,Apple,AAPL,Operating Cash Flow,118.254
4,2025,2024-09-29,2025-09-27,111482000000,2025-10-31,Apple,AAPL,Operating Cash Flow,111.482


In [76]:
capex_concepts = [
    key
    for key in us_gaap.keys()
    if "PaymentsToAcquire" in key
]

capex_concepts

['PaymentsToAcquireAvailableForSaleSecurities',
 'PaymentsToAcquireAvailableForSaleSecuritiesDebt',
 'PaymentsToAcquireBusinessesNetOfCashAcquired',
 'PaymentsToAcquireIntangibleAssets',
 'PaymentsToAcquireOtherInvestments',
 'PaymentsToAcquireProductiveAssets',
 'PaymentsToAcquirePropertyPlantAndEquipment']

In [77]:
capex_concept = us_gaap[
    "PaymentsToAcquirePropertyPlantAndEquipment"
]

In [78]:
print("Label:")
print(capex_concept["label"])

print("\nDescription:")
print(capex_concept["description"])

print("\nAvailable units:")
print(list(capex_concept["units"].keys()))

Label:
Payments to Acquire Property, Plant, and Equipment

Description:
The cash outflow associated with the acquisition of long-lived, physical assets that are used in the normal conduct of business to produce goods and services and not intended for resale; includes cash outflows to pay for construction of self-constructed assets.

Available units:
['USD']


In [79]:
apple_capex = extract_annual_metric(
    company_data=apple_data,
    concept_name="PaymentsToAcquirePropertyPlantAndEquipment",
    company="Apple",
    ticker="AAPL",
    metric="Capital Expenditure"
)

In [80]:
apple_capex

,fiscal_year,period_start,period_end,value_usd,filed,company,ticker,metric,value_billions
0,2021,2020-09-27,2021-09-25,11085000000,2021-10-29,Apple,AAPL,Capital Expenditure,11.085
1,2022,2021-09-26,2022-09-24,10708000000,2022-10-28,Apple,AAPL,Capital Expenditure,10.708
2,2023,2022-09-25,2023-09-30,10959000000,2023-11-03,Apple,AAPL,Capital Expenditure,10.959
3,2024,2023-10-01,2024-09-28,9447000000,2024-11-01,Apple,AAPL,Capital Expenditure,9.447
4,2025,2024-09-29,2025-09-27,12715000000,2025-10-31,Apple,AAPL,Capital Expenditure,12.715


In [81]:
apple_metrics = pd.concat(
    [
        test_revenue,
        apple_operating_income,
        apple_net_income,
        apple_rd,
        apple_operating_cash_flow,
        apple_capex
    ],
    ignore_index=True
)

In [82]:
apple_metrics

,fiscal_year,period_start,period_end,value_usd,filed,company,ticker,metric,value_billions
0,2021,2020-09-27,2021-09-25,365817000000,2021-10-29,Apple,AAPL,Revenue,365.817
1,2022,2021-09-26,2022-09-24,394328000000,2022-10-28,Apple,AAPL,Revenue,394.328
2,2023,2022-09-25,2023-09-30,383285000000,2023-11-03,Apple,AAPL,Revenue,383.285
3,2024,2023-10-01,2024-09-28,391035000000,2024-11-01,Apple,AAPL,Revenue,391.035
4,2025,2024-09-29,2025-09-27,416161000000,2025-10-31,Apple,AAPL,Revenue,416.161
5,2021,2020-09-27,2021-09-25,108949000000,2021-10-29,Apple,AAPL,Operating Income,108.949
6,2022,2021-09-26,2022-09-24,119437000000,2022-10-28,Apple,AAPL,Operating Income,119.437
7,2023,2022-09-25,2023-09-30,114301000000,2023-11-03,Apple,AAPL,Operating Income,114.301
8,2024,2023-10-01,2024-09-28,123216000000,2024-11-01,Apple,AAPL,Operating Income,123.216
9,2025,2024-09-29,2025-09-27,133050000000,2025-10-31,Apple,AAPL,Operating Income,133.050


In [83]:
apple_metrics["metric"].value_counts()

metric
Revenue                5
Operating Income       5
Net Income             5
R&D Expense            5
Operating Cash Flow    5
Capital Expenditure    5
Name: count, dtype: int64

In [84]:
apple_wide = (
    apple_metrics
    .pivot(
        index="fiscal_year",
        columns="metric",
        values="value_billions"
    )
    .reset_index()
)

In [85]:
apple_wide.columns.name = None

In [86]:
def calculate_derived_metrics(wide_df):
    df = wide_df.copy()

    df["Operating Margin (%)"] = (
        df["Operating Income"]
        / df["Revenue"]
        * 100
    ).round(2)

    df["Net Margin (%)"] = (
        df["Net Income"]
        / df["Revenue"]
        * 100
    ).round(2)

    df["R&D as % of Revenue"] = (
        df["R&D Expense"]
        / df["Revenue"]
        * 100
    ).round(2)

    df["Free Cash Flow"] = (
        df["Operating Cash Flow"]
        - df["Capital Expenditure"]
    )

    df["CapEx as % of Revenue"] = (
    df["Capital Expenditure"]
    / df["Revenue"]
    * 100
    ).round(2)

    df["Free Cash Flow Margin (%)"] = (
    df["Free Cash Flow"]
    / df["Revenue"]
    * 100
    ).round(2)

    df["Revenue Growth (%)"] = (
        df["Revenue"]
        .pct_change()
        .mul(100)
        .round(2)
    )

    return df

In [87]:
apple_wide = calculate_derived_metrics(apple_wide)

In [88]:
apple_wide

,fiscal_year,Capital Expenditure,Net Income,Operating Cash Flow,Operating Income,R&D Expense,Revenue,Operating Margin (%),Net Margin (%),R&D as % of Revenue,Free Cash Flow,CapEx as % of Revenue,Free Cash Flow Margin (%),Revenue Growth (%)
0,2021,11.085,94.680,104.038,108.949,21.914,365.817,29.78,25.88,5.99,92.953,3.03,25.41,NaN
1,2022,10.708,99.803,122.151,119.437,26.251,394.328,30.29,25.31,6.66,111.443,2.72,28.26,7.79
2,2023,10.959,96.995,110.543,114.301,29.915,383.285,29.82,25.31,7.80,99.584,2.86,25.98,-2.80
3,2024,9.447,93.736,118.254,123.216,31.370,391.035,31.51,23.97,8.02,108.807,2.42,27.83,2.02
4,2025,12.715,112.010,111.482,133.050,34.550,416.161,31.97,26.92,8.30,98.767,3.06,23.73,6.43


In [89]:
def fetch_company_facts(cik, headers):
    url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"

    response = requests.get(url, headers=headers)

    response.raise_for_status()

    return response.json()

In [90]:
MICROSOFT_CIK = "0000789019"

In [91]:
microsoft_data = fetch_company_facts(
    cik=MICROSOFT_CIK,
    headers=headers
)

In [92]:
microsoft_data["entityName"]

'MICROSOFT CORPORATION'

In [93]:
microsoft_data["cik"]

789019

In [94]:
microsoft_us_gaap = microsoft_data["facts"]["us-gaap"]

In [95]:
len(microsoft_us_gaap)

562

In [96]:
microsoft_revenue_concepts = [
    key
    for key in microsoft_us_gaap.keys()
    if "Revenue" in key
]

microsoft_revenue_concepts

['BusinessAcquisitionsProFormaRevenue',
 'ContractWithCustomerLiabilityRevenueRecognized',
 'CostOfRevenue',
 'DeferredRevenue',
 'DeferredRevenueAdditions',
 'DeferredRevenueCurrent',
 'DeferredRevenueNoncurrent',
 'DeferredRevenueRevenueRecognized',
 'DeferredRevenueRevenueRecognized1',
 'EntityWideDisclosureOnGeographicAreasRevenueFromExternalCustomersAttributedToEntitysCountryOfDomicile',
 'EntityWideDisclosureOnGeographicAreasRevenueFromExternalCustomersAttributedToForeignCountries',
 'IncreaseDecreaseInDeferredRevenue',
 'RecognitionOfDeferredRevenue',
 'RevenueFromContractWithCustomerExcludingAssessedTax',
 'RevenueRemainingPerformanceObligation',
 'RevenueRemainingPerformanceObligationPercentage',
 'Revenues',
 'SalesRevenueGoodsNet',
 'SalesRevenueNet']

In [97]:
microsoft_revenue_concept = microsoft_us_gaap[
    "RevenueFromContractWithCustomerExcludingAssessedTax"
]

In [98]:
print("Label:")
print(microsoft_revenue_concept["label"])

print("\nDescription:")
print(microsoft_revenue_concept["description"])

print("\nAvailable units:")
print(list(microsoft_revenue_concept["units"].keys()))

Label:
Revenue from Contract with Customer, Excluding Assessed Tax

Description:
Amount, excluding tax collected from customer, of revenue from satisfaction of performance obligation by transferring promised good or service to customer. Tax collected from customer is tax assessed by governmental authority that is both imposed on and concurrent with specific revenue-producing transaction, including, but not limited to, sales, use, value added and excise.

Available units:
['USD']


In [99]:
microsoft_revenue = extract_annual_metric(
    company_data=microsoft_data,
    concept_name="RevenueFromContractWithCustomerExcludingAssessedTax",
    company="Microsoft",
    ticker="MSFT",
    metric="Revenue"
)

In [100]:
microsoft_revenue

,fiscal_year,period_start,period_end,value_usd,filed,company,ticker,metric,value_billions
0,2021,2020-07-01,2021-06-30,168088000000,2021-07-29,Microsoft,MSFT,Revenue,168.088
1,2022,2021-07-01,2022-06-30,198270000000,2022-07-28,Microsoft,MSFT,Revenue,198.270
2,2023,2022-07-01,2023-06-30,211915000000,2023-07-27,Microsoft,MSFT,Revenue,211.915
3,2024,2023-07-01,2024-06-30,245122000000,2024-07-30,Microsoft,MSFT,Revenue,245.122
4,2025,2024-07-01,2025-06-30,281724000000,2025-07-30,Microsoft,MSFT,Revenue,281.724


In [101]:
microsoft_operating_income_concepts = [
    key
    for key in microsoft_us_gaap.keys()
    if "OperatingIncome" in key
]

microsoft_operating_income_concepts

['OperatingIncomeLoss']

In [102]:
microsoft_operating_income_concept = microsoft_us_gaap[
    "OperatingIncomeLoss"
]

In [103]:
print("Label:")
print(microsoft_operating_income_concept["label"])

print("\nDescription:")
print(microsoft_operating_income_concept["description"])

print("\nAvailable units:")
print(list(microsoft_operating_income_concept["units"].keys()))

Label:
Operating Income (Loss)

Description:
The net result for the period of deducting operating expenses from operating revenues.

Available units:
['USD']


In [104]:
microsoft_operating_income = extract_annual_metric(
    company_data=microsoft_data,
    concept_name="OperatingIncomeLoss",
    company="Microsoft",
    ticker="MSFT",
    metric="Operating Income"
)

In [105]:
microsoft_operating_income

,fiscal_year,period_start,period_end,value_usd,filed,company,ticker,metric,value_billions
0,2021,2020-07-01,2021-06-30,69916000000,2021-07-29,Microsoft,MSFT,Operating Income,69.916
1,2022,2021-07-01,2022-06-30,83383000000,2022-07-28,Microsoft,MSFT,Operating Income,83.383
2,2023,2022-07-01,2023-06-30,88523000000,2023-07-27,Microsoft,MSFT,Operating Income,88.523
3,2024,2023-07-01,2024-06-30,109433000000,2024-07-30,Microsoft,MSFT,Operating Income,109.433
4,2025,2024-07-01,2025-06-30,128528000000,2025-07-30,Microsoft,MSFT,Operating Income,128.528


In [106]:
[
    "NetIncomeLoss" in microsoft_us_gaap,
    "ResearchAndDevelopmentExpense" in microsoft_us_gaap,
    "NetCashProvidedByUsedInOperatingActivities" in microsoft_us_gaap,
    "PaymentsToAcquirePropertyPlantAndEquipment" in microsoft_us_gaap
]

[True, True, True, True]

In [107]:
microsoft_net_income = extract_annual_metric(
    company_data=microsoft_data,
    concept_name="NetIncomeLoss",
    company="Microsoft",
    ticker="MSFT",
    metric="Net Income"
)

microsoft_rd = extract_annual_metric(
    company_data=microsoft_data,
    concept_name="ResearchAndDevelopmentExpense",
    company="Microsoft",
    ticker="MSFT",
    metric="R&D Expense"
)

microsoft_operating_cash_flow = extract_annual_metric(
    company_data=microsoft_data,
    concept_name="NetCashProvidedByUsedInOperatingActivities",
    company="Microsoft",
    ticker="MSFT",
    metric="Operating Cash Flow"
)

microsoft_capex = extract_annual_metric(
    company_data=microsoft_data,
    concept_name="PaymentsToAcquirePropertyPlantAndEquipment",
    company="Microsoft",
    ticker="MSFT",
    metric="Capital Expenditure"
)

In [108]:
microsoft_metrics = pd.concat(
    [
        microsoft_revenue,
        microsoft_operating_income,
        microsoft_net_income,
        microsoft_rd,
        microsoft_operating_cash_flow,
        microsoft_capex
    ],
    ignore_index=True
)

In [109]:
microsoft_metrics["metric"].value_counts()

metric
Revenue                5
Operating Income       5
Net Income             5
R&D Expense            5
Operating Cash Flow    5
Capital Expenditure    5
Name: count, dtype: int64

In [110]:
microsoft_wide = (
    microsoft_metrics
    .pivot(
        index="fiscal_year",
        columns="metric",
        values="value_billions"
    )
    .reset_index()
)

microsoft_wide.columns.name = None

In [111]:
microsoft_wide = calculate_derived_metrics(microsoft_wide)

In [112]:
microsoft_wide

,fiscal_year,Capital Expenditure,Net Income,Operating Cash Flow,Operating Income,R&D Expense,Revenue,Operating Margin (%),Net Margin (%),R&D as % of Revenue,Free Cash Flow,CapEx as % of Revenue,Free Cash Flow Margin (%),Revenue Growth (%)
0,2021,20.622,61.271,76.740,69.916,20.716,168.088,41.59,36.45,12.32,56.118,12.27,33.39,NaN
1,2022,23.886,72.738,89.035,83.383,24.512,198.270,42.06,36.69,12.36,65.149,12.05,32.86,17.96
2,2023,28.107,72.361,87.582,88.523,27.195,211.915,41.77,34.15,12.83,59.475,13.26,28.07,6.88
3,2024,44.477,88.136,118.548,109.433,29.510,245.122,44.64,35.96,12.04,74.071,18.14,30.22,15.67
4,2025,64.551,101.832,136.162,128.528,32.488,281.724,45.62,36.15,11.53,71.611,22.91,25.42,14.93


In [113]:
ALPHABET_CIK = "0001652044"

In [114]:
alphabet_data = fetch_company_facts(
    cik=ALPHABET_CIK,
    headers=headers
)

In [115]:
alphabet_data["entityName"]

'Alphabet Inc.'

In [116]:
alphabet_us_gaap = alphabet_data["facts"]["us-gaap"]

In [117]:
len(alphabet_us_gaap)

543

In [118]:
alphabet_revenue_concepts = [
    key
    for key in alphabet_us_gaap.keys()
    if "Revenue" in key
]

alphabet_revenue_concepts

['ContractWithCustomerLiabilityRevenueRecognized',
 'CostOfRevenue',
 'DeferredRevenueCurrent',
 'DeferredRevenueNoncurrent',
 'DeferredRevenueRevenueRecognized1',
 'IncreaseDecreaseInDeferredRevenue',
 'RevenueFromContractWithCustomerExcludingAssessedTax',
 'RevenueRemainingPerformanceObligation',
 'Revenues',
 'RevenueNotFromContractWithCustomer']

In [119]:
alphabet_revenue_concept = alphabet_us_gaap[
    "RevenueFromContractWithCustomerExcludingAssessedTax"
]

In [120]:
print("Label:")
print(alphabet_revenue_concept["label"])

print("\nDescription:")
print(alphabet_revenue_concept["description"])

print("\nAvailable units:")
print(list(alphabet_revenue_concept["units"].keys()))

Label:
Revenue from Contract with Customer, Excluding Assessed Tax

Description:
Amount, excluding tax collected from customer, of revenue from satisfaction of performance obligation by transferring promised good or service to customer. Tax collected from customer is tax assessed by governmental authority that is both imposed on and concurrent with specific revenue-producing transaction, including, but not limited to, sales, use, value added and excise.

Available units:
['USD']


In [121]:
alphabet_revenue = extract_annual_metric(
    company_data=alphabet_data,
    concept_name=[
        "RevenueFromContractWithCustomerExcludingAssessedTax",
        "Revenues"
    ],
    company="Alphabet",
    ticker="GOOGL",
    metric="Revenue"
)

In [122]:
alphabet_revenue

,fiscal_year,period_start,period_end,value_usd,filed,company,ticker,metric,value_billions
0,2021,2021-01-01,2021-12-31,257637000000,2022-02-02,Alphabet,GOOGL,Revenue,257.637
1,2022,2022-01-01,2022-12-31,282836000000,2023-02-03,Alphabet,GOOGL,Revenue,282.836
2,2023,2023-01-01,2023-12-31,307394000000,2024-01-31,Alphabet,GOOGL,Revenue,307.394
3,2024,2024-01-01,2024-12-31,350018000000,2025-02-05,Alphabet,GOOGL,Revenue,350.018
4,2025,2025-01-01,2025-12-31,402836000000,2026-02-05,Alphabet,GOOGL,Revenue,402.836


In [123]:
[
    "OperatingIncomeLoss" in alphabet_us_gaap,
    "NetIncomeLoss" in alphabet_us_gaap,
    "ResearchAndDevelopmentExpense" in alphabet_us_gaap,
    "NetCashProvidedByUsedInOperatingActivities" in alphabet_us_gaap,
    "PaymentsToAcquirePropertyPlantAndEquipment" in alphabet_us_gaap
]

[True, True, True, True, True]

In [124]:
alphabet_revenues_alt = pd.DataFrame(
    alphabet_us_gaap["Revenues"]["units"]["USD"]
)

alphabet_revenues_alt["start"] = pd.to_datetime(
    alphabet_revenues_alt["start"]
)

alphabet_revenues_alt["end"] = pd.to_datetime(
    alphabet_revenues_alt["end"]
)

In [125]:
alphabet_revenues_alt[
    (alphabet_revenues_alt["form"] == "10-K") &
    (alphabet_revenues_alt["fp"] == "FY") &
    (alphabet_revenues_alt["end"].dt.year >= 2023)
][
    ["start", "end", "val", "fy", "filed", "frame"]
]

,start,end,val,fy,filed,frame
60,2023-01-01,2023-12-31,307394000000,2025,2026-02-05,CY2023
65,2024-01-01,2024-12-31,350018000000,2025,2026-02-05,CY2024
73,2025-01-01,2025-12-31,402836000000,2025,2026-02-05,CY2025


In [126]:
alphabet_operating_income = extract_annual_metric(
    company_data=alphabet_data,
    concept_name="OperatingIncomeLoss",
    company="Alphabet",
    ticker="GOOGL",
    metric="Operating Income"
)

alphabet_net_income = extract_annual_metric(
    company_data=alphabet_data,
    concept_name="NetIncomeLoss",
    company="Alphabet",
    ticker="GOOGL",
    metric="Net Income"
)

alphabet_rd = extract_annual_metric(
    company_data=alphabet_data,
    concept_name="ResearchAndDevelopmentExpense",
    company="Alphabet",
    ticker="GOOGL",
    metric="R&D Expense"
)

alphabet_operating_cash_flow = extract_annual_metric(
    company_data=alphabet_data,
    concept_name="NetCashProvidedByUsedInOperatingActivities",
    company="Alphabet",
    ticker="GOOGL",
    metric="Operating Cash Flow"
)

alphabet_capex = extract_annual_metric(
    company_data=alphabet_data,
    concept_name="PaymentsToAcquirePropertyPlantAndEquipment",
    company="Alphabet",
    ticker="GOOGL",
    metric="Capital Expenditure"
)

In [127]:
alphabet_metric_counts = {
    "Revenue": len(alphabet_revenue),
    "Operating Income": len(alphabet_operating_income),
    "Net Income": len(alphabet_net_income),
    "R&D Expense": len(alphabet_rd),
    "Operating Cash Flow": len(alphabet_operating_cash_flow),
    "Capital Expenditure": len(alphabet_capex)
}

alphabet_metric_counts

{'Revenue': 5,
 'Operating Income': 5,
 'Net Income': 5,
 'R&D Expense': 5,
 'Operating Cash Flow': 5,
 'Capital Expenditure': 5}

In [128]:
alphabet_metrics = pd.concat(
    [
        alphabet_revenue,
        alphabet_operating_income,
        alphabet_net_income,
        alphabet_rd,
        alphabet_operating_cash_flow,
        alphabet_capex
    ],
    ignore_index=True
)

In [129]:
alphabet_metrics["metric"].value_counts()

metric
Revenue                5
Operating Income       5
Net Income             5
R&D Expense            5
Operating Cash Flow    5
Capital Expenditure    5
Name: count, dtype: int64

In [130]:
alphabet_wide = (
    alphabet_metrics
    .pivot(
        index="fiscal_year",
        columns="metric",
        values="value_billions"
    )
    .reset_index()
)

alphabet_wide.columns.name = None

alphabet_wide = calculate_derived_metrics(alphabet_wide)

In [131]:
alphabet_wide

,fiscal_year,Capital Expenditure,Net Income,Operating Cash Flow,Operating Income,R&D Expense,Revenue,Operating Margin (%),Net Margin (%),R&D as % of Revenue,Free Cash Flow,CapEx as % of Revenue,Free Cash Flow Margin (%),Revenue Growth (%)
0,2021,24.640,76.033,91.652,78.714,31.562,257.637,30.55,29.51,12.25,67.012,9.56,26.01,NaN
1,2022,31.485,59.972,91.495,74.842,39.500,282.836,26.46,21.20,13.97,60.010,11.13,21.22,9.78
2,2023,32.251,73.795,101.746,84.293,45.427,307.394,27.42,24.01,14.78,69.495,10.49,22.61,8.68
3,2024,52.535,100.118,125.299,112.390,49.326,350.018,32.11,28.60,14.09,72.764,15.01,20.79,13.87
4,2025,91.447,132.170,164.713,129.039,61.087,402.836,32.03,32.81,15.16,73.266,22.70,18.19,15.09


In [132]:
apple_wide["company"] = "Apple"
apple_wide["ticker"] = "AAPL"

microsoft_wide["company"] = "Microsoft"
microsoft_wide["ticker"] = "MSFT"

alphabet_wide["company"] = "Alphabet"
alphabet_wide["ticker"] = "GOOGL"

In [133]:
company_kpis = pd.concat(
    [
        apple_wide,
        microsoft_wide,
        alphabet_wide
    ],
    ignore_index=True
)

In [134]:
company_kpis[
    [
        "company",
        "fiscal_year",
        "Revenue",
        "Operating Margin (%)",
        "Net Margin (%)",
        "Revenue Growth (%)"
    ]
]

,company,fiscal_year,Revenue,Operating Margin (%),Net Margin (%),Revenue Growth (%)
0,Apple,2021,365.817,29.78,25.88,NaN
1,Apple,2022,394.328,30.29,25.31,7.79
2,Apple,2023,383.285,29.82,25.31,-2.80
3,Apple,2024,391.035,31.51,23.97,2.02
4,Apple,2025,416.161,31.97,26.92,6.43
5,Microsoft,2021,168.088,41.59,36.45,NaN
6,Microsoft,2022,198.270,42.06,36.69,17.96
7,Microsoft,2023,211.915,41.77,34.15,6.88
8,Microsoft,2024,245.122,44.64,35.96,15.67
9,Microsoft,2025,281.724,45.62,36.15,14.93


In [135]:
financial_metrics_long = pd.concat(
    [
        apple_metrics,
        microsoft_metrics,
        alphabet_metrics
    ],
    ignore_index=True
)

In [136]:
financial_metrics_long.groupby(
    ["company", "metric"]
).size()

company    metric             
Alphabet   Capital Expenditure    5
           Net Income             5
           Operating Cash Flow    5
           Operating Income       5
           R&D Expense            5
           Revenue                5
Apple      Capital Expenditure    5
           Net Income             5
           Operating Cash Flow    5
           Operating Income       5
           R&D Expense            5
           Revenue                5
Microsoft  Capital Expenditure    5
           Net Income             5
           Operating Cash Flow    5
           Operating Income       5
           R&D Expense            5
           Revenue                5
dtype: int64

In [137]:
latest_2025 = company_kpis[
    company_kpis["fiscal_year"] == 2025
].copy()

In [138]:
latest_2025[
    [
        "company",
        "Revenue",
        "Operating Income",
        "Net Income",
        "Operating Margin (%)",
        "Net Margin (%)",
        "R&D as % of Revenue",
        "Free Cash Flow",
        "Revenue Growth (%)"
    ]
]

,company,Revenue,Operating Income,Net Income,Operating Margin (%),Net Margin (%),R&D as % of Revenue,Free Cash Flow,Revenue Growth (%)
4,Apple,416.161,133.050,112.010,31.97,26.92,8.30,98.767,6.43
9,Microsoft,281.724,128.528,101.832,45.62,36.15,11.53,71.611,14.93
14,Alphabet,402.836,129.039,132.170,32.03,32.81,15.16,73.266,15.09


In [139]:
company_kpis.groupby("company")["fiscal_year"].nunique()

company
Alphabet     5
Apple        5
Microsoft    5
Name: fiscal_year, dtype: int64

In [140]:
cagr_data = (
    company_kpis
    .sort_values(["company", "fiscal_year"])
    .groupby("company")
    .agg(
        start_revenue=("Revenue", "first"),
        end_revenue=("Revenue", "last"),
        start_year=("fiscal_year", "first"),
        end_year=("fiscal_year", "last")
    )
    .reset_index()
)

In [141]:
cagr_data

,company,start_revenue,end_revenue,start_year,end_year
0,Alphabet,257.637,402.836,2021,2025
1,Apple,365.817,416.161,2021,2025
2,Microsoft,168.088,281.724,2021,2025


In [142]:
cagr_data["years"] = (
    cagr_data["end_year"]
    - cagr_data["start_year"]
)

In [143]:
cagr_data["Revenue CAGR (%)"] = (
    (
        (
            cagr_data["end_revenue"]
            / cagr_data["start_revenue"]
        )
        ** (1 / cagr_data["years"])
        - 1
    )
    * 100
).round(2)

In [144]:
cagr_data[
    [
        "company",
        "start_revenue",
        "end_revenue",
        "Revenue CAGR (%)"
    ]
]

,company,start_revenue,end_revenue,Revenue CAGR (%)
0,Alphabet,257.637,402.836,11.82
1,Apple,365.817,416.161,3.28
2,Microsoft,168.088,281.724,13.78


In [145]:
margin_change = (
    company_kpis
    .sort_values(["company", "fiscal_year"])
    .groupby("company")
    .agg(
        start_operating_margin=("Operating Margin (%)", "first"),
        end_operating_margin=("Operating Margin (%)", "last")
    )
    .reset_index()
)

In [146]:
margin_change["Operating Margin Change (pp)"] = (
    margin_change["end_operating_margin"]
    - margin_change["start_operating_margin"]
).round(2)

In [147]:
margin_change

,company,start_operating_margin,end_operating_margin,Operating Margin Change (pp)
0,Alphabet,30.55,32.03,1.48
1,Apple,29.78,31.97,2.19
2,Microsoft,41.59,45.62,4.03


In [148]:
latest_summary = latest_2025[
    [
        "company",
        "Revenue",
        "Operating Margin (%)",
        "Net Margin (%)",
        "R&D as % of Revenue",
        "Free Cash Flow",
        "Revenue Growth (%)"
    ]
].copy()

In [149]:
executive_summary = latest_summary.merge(
    cagr_data[
        ["company", "Revenue CAGR (%)"]
    ],
    on="company",
    how="left"
)

In [150]:
executive_summary = executive_summary.merge(
    margin_change[
        ["company", "Operating Margin Change (pp)"]
    ],
    on="company",
    how="left"
)

In [151]:
executive_summary

,company,Revenue,Operating Margin (%),Net Margin (%),R&D as % of Revenue,Free Cash Flow,Revenue Growth (%),Revenue CAGR (%),Operating Margin Change (pp)
0,Apple,416.161,31.97,26.92,8.30,98.767,6.43,3.28,2.19
1,Microsoft,281.724,45.62,36.15,11.53,71.611,14.93,13.78,4.03
2,Alphabet,402.836,32.03,32.81,15.16,73.266,15.09,11.82,1.48


In [152]:
growth_profitability = executive_summary[
    [
        "company",
        "Revenue CAGR (%)",
        "Revenue Growth (%)",
        "Operating Margin (%)",
        "Operating Margin Change (pp)",
        "Net Margin (%)"
    ]
].copy()

In [153]:
growth_profitability = growth_profitability.sort_values(
    "Revenue CAGR (%)",
    ascending=False
).reset_index(drop=True)

In [154]:
growth_profitability

,company,Revenue CAGR (%),Revenue Growth (%),Operating Margin (%),Operating Margin Change (pp),Net Margin (%)
0,Microsoft,13.78,14.93,45.62,4.03,36.15
1,Alphabet,11.82,15.09,32.03,1.48,32.81
2,Apple,3.28,6.43,31.97,2.19,26.92


In [155]:
rd_trend = company_kpis[
    [
        "company",
        "fiscal_year",
        "R&D Expense",
        "Revenue",
        "R&D as % of Revenue"
    ]
].copy()

rd_trend = rd_trend.sort_values(
    ["company", "fiscal_year"]
).reset_index(drop=True)

In [156]:
rd_trend

,company,fiscal_year,R&D Expense,Revenue,R&D as % of Revenue
0,Alphabet,2021,31.562,257.637,12.25
1,Alphabet,2022,39.500,282.836,13.97
2,Alphabet,2023,45.427,307.394,14.78
3,Alphabet,2024,49.326,350.018,14.09
4,Alphabet,2025,61.087,402.836,15.16
5,Apple,2021,21.914,365.817,5.99
6,Apple,2022,26.251,394.328,6.66
7,Apple,2023,29.915,383.285,7.80
8,Apple,2024,31.370,391.035,8.02
9,Apple,2025,34.550,416.161,8.30


In [157]:
rd_intensity_change = (
    rd_trend
    .groupby("company")
    .agg(
        start_rd_intensity=("R&D as % of Revenue", "first"),
        end_rd_intensity=("R&D as % of Revenue", "last")
    )
    .reset_index()
)

rd_intensity_change["R&D Intensity Change (pp)"] = (
    rd_intensity_change["end_rd_intensity"]
    - rd_intensity_change["start_rd_intensity"]
).round(2)

In [158]:
rd_intensity_change

,company,start_rd_intensity,end_rd_intensity,R&D Intensity Change (pp)
0,Alphabet,12.25,15.16,2.91
1,Apple,5.99,8.30,2.31
2,Microsoft,12.32,11.53,-0.79


In [159]:
rd_cagr = (
    company_kpis
    .sort_values(["company", "fiscal_year"])
    .groupby("company")
    .agg(
        start_rd=("R&D Expense", "first"),
        end_rd=("R&D Expense", "last"),
        start_year=("fiscal_year", "first"),
        end_year=("fiscal_year", "last")
    )
    .reset_index()
)

In [160]:
rd_cagr["years"] = (
    rd_cagr["end_year"]
    - rd_cagr["start_year"]
)

In [161]:
rd_cagr["R&D CAGR (%)"] = (
    (
        rd_cagr["end_rd"]
        / rd_cagr["start_rd"]
    )
    ** (1 / rd_cagr["years"])
    - 1
) * 100

In [162]:
rd_cagr["R&D CAGR (%)"] = (
    rd_cagr["R&D CAGR (%)"].round(2)
)

In [163]:
rd_cagr[
    [
        "company",
        "start_rd",
        "end_rd",
        "R&D CAGR (%)"
    ]
]

,company,start_rd,end_rd,R&D CAGR (%)
0,Alphabet,31.562,61.087,17.95
1,Apple,21.914,34.550,12.06
2,Microsoft,20.716,32.488,11.91


In [164]:
company_kpis = pd.concat(
    [
        apple_wide,
        microsoft_wide,
        alphabet_wide
    ],
    ignore_index=True
)

In [165]:
company_kpis[
    [
        "company",
        "fiscal_year",
        "Revenue",
        "Free Cash Flow",
        "Free Cash Flow Margin (%)"
    ]
]

,company,fiscal_year,Revenue,Free Cash Flow,Free Cash Flow Margin (%)
0,Apple,2021,365.817,92.953,25.41
1,Apple,2022,394.328,111.443,28.26
2,Apple,2023,383.285,99.584,25.98
3,Apple,2024,391.035,108.807,27.83
4,Apple,2025,416.161,98.767,23.73
5,Microsoft,2021,168.088,56.118,33.39
6,Microsoft,2022,198.270,65.149,32.86
7,Microsoft,2023,211.915,59.475,28.07
8,Microsoft,2024,245.122,74.071,30.22
9,Microsoft,2025,281.724,71.611,25.42


In [166]:
fcf_margin_change = (
    company_kpis
    .sort_values(["company", "fiscal_year"])
    .groupby("company")
    .agg(
        start_fcf_margin=("Free Cash Flow Margin (%)", "first"),
        end_fcf_margin=("Free Cash Flow Margin (%)", "last")
    )
    .reset_index()
)

In [167]:
fcf_margin_change["FCF Margin Change (pp)"] = (
    fcf_margin_change["end_fcf_margin"]
    - fcf_margin_change["start_fcf_margin"]
).round(2)

In [168]:
fcf_margin_change

,company,start_fcf_margin,end_fcf_margin,FCF Margin Change (pp)
0,Alphabet,26.01,18.19,-7.82
1,Apple,25.41,23.73,-1.68
2,Microsoft,33.39,25.42,-7.97


In [169]:
company_kpis[
    [
        "company",
        "fiscal_year",
        "Capital Expenditure",
        "CapEx as % of Revenue",
        "Free Cash Flow Margin (%)"
    ]
]

,company,fiscal_year,Capital Expenditure,CapEx as % of Revenue,Free Cash Flow Margin (%)
0,Apple,2021,11.085,3.03,25.41
1,Apple,2022,10.708,2.72,28.26
2,Apple,2023,10.959,2.86,25.98
3,Apple,2024,9.447,2.42,27.83
4,Apple,2025,12.715,3.06,23.73
5,Microsoft,2021,20.622,12.27,33.39
6,Microsoft,2022,23.886,12.05,32.86
7,Microsoft,2023,28.107,13.26,28.07
8,Microsoft,2024,44.477,18.14,30.22
9,Microsoft,2025,64.551,22.91,25.42


In [170]:
capex_intensity_change = (
    company_kpis
    .sort_values(["company", "fiscal_year"])
    .groupby("company")
    .agg(
        start_capex_intensity=("CapEx as % of Revenue", "first"),
        end_capex_intensity=("CapEx as % of Revenue", "last")
    )
    .reset_index()
)

In [171]:
capex_intensity_change["CapEx Intensity Change (pp)"] = (
    capex_intensity_change["end_capex_intensity"]
    - capex_intensity_change["start_capex_intensity"]
).round(2)

In [172]:
capex_intensity_change

,company,start_capex_intensity,end_capex_intensity,CapEx Intensity Change (pp)
0,Alphabet,9.56,22.70,13.14
1,Apple,3.03,3.06,0.03
2,Microsoft,12.27,22.91,10.64


In [173]:
cash_investment_summary = capex_intensity_change.merge(
    fcf_margin_change[
        ["company", "FCF Margin Change (pp)"]
    ],
    on="company",
    how="left"
)

cash_investment_summary

,company,start_capex_intensity,end_capex_intensity,CapEx Intensity Change (pp),FCF Margin Change (pp)
0,Alphabet,9.56,22.70,13.14,-7.82
1,Apple,3.03,3.06,0.03,-1.68
2,Microsoft,12.27,22.91,10.64,-7.97


In [174]:
from pathlib import Path

Path.cwd()

WindowsPath('d:/Projects/big-tech-financial-analytics/notebooks')

In [175]:
from pathlib import Path

project_root = Path.cwd().parent
output_dir = project_root / "data" / "processed"

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

In [176]:
output_dir

WindowsPath('d:/Projects/big-tech-financial-analytics/data/processed')

In [177]:
financial_metrics_long.to_csv(
    output_dir / "financial_metrics_long.csv",
    index=False
)

company_kpis.to_csv(
    output_dir / "company_kpis.csv",
    index=False
)

In [178]:
list(output_dir.iterdir())

[WindowsPath('d:/Projects/big-tech-financial-analytics/data/processed/company_kpis.csv'),
 WindowsPath('d:/Projects/big-tech-financial-analytics/data/processed/financial_metrics_long.csv')]

In [179]:
check_long = pd.read_csv(
    output_dir / "financial_metrics_long.csv"
)

check_kpis = pd.read_csv(
    output_dir / "company_kpis.csv"
)